# 02 — Preparação dos Dados

## Pergunta de investigação

> **Conseguimos transformar esses dados num dataset utilizável?**

Neste notebook será realizada a preparação dos dados recolhidos.

### Objetivos

- Limpar os dados;
- Corrigir ou tratar valores em falta;
- Normalizar formatos;
- Organizar as séries temporais;
- Criar o dataset final para análise e modelação.

In [18]:
import pandas as pd
from pathlib import Path

DATA_PROCESSED = Path("../data/processed")

materials_index_path = DATA_PROCESSED / "materials_index.csv"

materials_history = pd.read_csv(
    materials_index_path,
    encoding="utf-8"
)

materials_history.head()

,ano,codigo,material,mes,indice,ficheiro
0,2015,M01,Britas,JAN,104.4,cife2015_MATERIAIS.xls
1,2015,M01,Britas,FEV,104.7,cife2015_MATERIAIS.xls
2,2015,M01,Britas,MAR,105.0,cife2015_MATERIAIS.xls
3,2015,M01,Britas,ABR,104.8,cife2015_MATERIAIS.xls
4,2015,M01,Britas,MAI,104.5,cife2015_MATERIAIS.xls


In [19]:
print("Dimensão:", materials_history.shape)
print("\nColunas:")
print(materials_history.columns.tolist())

print("\nTipos:")
print(materials_history.dtypes)

Dimensão: (7644, 6)

Colunas:
['ano', 'codigo', 'material', 'mes', 'indice', 'ficheiro']

Tipos:
ano           int64
codigo          str
material        str
mes             str
indice      float64
ficheiro        str
dtype: object


In [20]:
materials_history.isna().sum()

ano           0
codigo        0
material      0
mes           0
indice      290
ficheiro      0
dtype: int64

In [21]:
duplicados = materials_history.duplicated(
    subset=["ano", "codigo", "mes"]
).sum()

print(f"Registos duplicados: {duplicados}")

Registos duplicados: 0


In [22]:
meses = {
    "JAN": 1,
    "FEV": 2,
    "MAR": 3,
    "ABR": 4,
    "MAI": 5,
    "JUN": 6,
    "JUL": 7,
    "AGO": 8,
    "SET": 9,
    "OUT": 10,
    "NOV": 11,
    "DEZ": 12
}

materials_history["mes_num"] = (
    materials_history["mes"]
    .str.upper()
    .str.strip()
    .map(meses)
)

In [23]:
print(
    materials_history[
        ["ano", "mes", "mes_num"]
    ].drop_duplicates().head(15)
)

      ano  mes  mes_num
0    2015  JAN        1
1    2015  FEV        2
2    2015  MAR        3
3    2015  ABR        4
4    2015  MAI        5
5    2015  JUN        6
6    2015  JUL        7
7    2015  AGO        8
8    2015  SET        9
9    2015  OUT       10
10   2015  NOV       11
11   2015  DEZ       12
588  2016  JAN        1
589  2016  FEV        2
590  2016  MAR        3


In [24]:
materials_history["data"] = pd.to_datetime(
    dict(
        year=materials_history["ano"],
        month=materials_history["mes_num"],
        day=1
    )
)

materials_history.head()

,ano,codigo,material,mes,indice,ficheiro,mes_num,data
0,2015,M01,Britas,JAN,104.4,cife2015_MATERIAIS.xls,1,2015-01-01
1,2015,M01,Britas,FEV,104.7,cife2015_MATERIAIS.xls,2,2015-02-01
2,2015,M01,Britas,MAR,105.0,cife2015_MATERIAIS.xls,3,2015-03-01
3,2015,M01,Britas,ABR,104.8,cife2015_MATERIAIS.xls,4,2015-04-01
4,2015,M01,Britas,MAI,104.5,cife2015_MATERIAIS.xls,5,2015-05-01


In [25]:
coverage = (
    materials_history
    .groupby(["ano", "codigo"])
    .size()
    .unstack(fill_value=0)
)

coverage.head()

codigo,M01,M02,M03,M04,M05,M06,M07,M08,M09,M10,...,M51,M52,M53,M54,M55,M56,M57,M58,M59,M60
ano,,,,,,,,,,,,,,,,,,,,,
2015,12,12,12,12,12,12,12,12,12,12,...,12,0,0,0,0,0,0,0,0,0
2016,12,12,12,12,12,12,12,12,12,12,...,12,0,0,0,0,0,0,0,0,0
2017,12,12,12,12,12,12,12,12,12,12,...,12,0,0,0,0,0,0,0,0,0
2018,12,12,12,12,12,12,12,12,12,12,...,12,0,0,0,0,0,0,0,0,0
2019,12,12,12,12,12,12,12,12,12,12,...,12,0,0,0,0,0,0,0,0,0


In [26]:
incompletos = (
    materials_history
    .groupby(["ano", "codigo"])
    .size()
    .reset_index(name="meses")
)

incompletos[incompletos["meses"] != 12].head(20)

,ano,codigo,meses


In [27]:
materials_prepared_path = (
    DATA_PROCESSED / "materials_history.csv"
)

materials_history.to_csv(
    materials_prepared_path,
    index=False,
    encoding="utf-8"
)

print(f"Dataset preparado guardado em:")
print(materials_prepared_path)

Dataset preparado guardado em:
..\data\processed\materials_history.csv


In [28]:
material_coverage = (
    materials_history
    .groupby(["codigo", "material"])
    .agg(
        primeiro_ano=("ano", "min"),
        ultimo_ano=("ano", "max"),
        anos=("ano", "nunique"),
        registos=("indice", "count")
    )
    .reset_index()
    .sort_values("codigo")
)

material_coverage

,codigo,material,primeiro_ano,ultimo_ano,anos,registos
0,M01,Britas,2015,2026,12,139
1,M02,Areias,2015,2026,12,139
2,M03,Inertes,2015,2026,12,139
3,M04,Ladrilhos de calcário e granito,2015,2026,12,139
4,M05,Cantarias de calcário e granito,2015,2026,12,139
5,M06,Ladr. e cant. de calcário e granito,2015,2026,12,139
6,M07,Telhas cerâmicas,2015,2026,12,139
7,M08,Tijolos cerâmicos,2015,2026,12,139
8,M09,Produtos Cerâmicos Vermelhos,2015,2026,12,139
9,M10,Azulejos e Mosaicos,2015,2026,12,139


In [29]:
material_start = (
    materials_history
    .groupby("codigo")
    .agg(
        primeiro_ano=("ano", "min"),
        material=("material", "first")
    )
    .reset_index()
)

material_start

,codigo,primeiro_ano,material
0,M01,2015,Britas
1,M02,2015,Areias
2,M03,2015,Inertes
3,M04,2015,Ladrilhos de calcário e granito
4,M05,2015,Cantarias de calcário e granito
5,M06,2015,Ladr. e cant. de calcário e granito
6,M07,2015,Telhas cerâmicas
7,M08,2015,Tijolos cerâmicos
8,M09,2015,Produtos Cerâmicos Vermelhos
9,M10,2015,Azulejos e Mosaicos


In [30]:
import json

materials_catalog_path = (
    DATA_PROCESSED / "materials_catalog.json"
)

with open(
    materials_catalog_path,
    "r",
    encoding="utf-8"
) as f:
    materials_catalog = pd.DataFrame(json.load(f))

print(f"Materiais no catálogo: {len(materials_catalog)}")
materials_catalog.head()

Materiais no catálogo: 58


,codigo,descricao
0,M01,Britas
1,M02,Areias
2,M03,Inertes
3,M04,Ladrilhos de calcário e granito
4,M05,Cantarias de calcário e granito


In [31]:
catalog_codes = set(materials_catalog["codigo"])

history_codes = set(materials_history["codigo"])

print("No catálogo mas sem histórico:")
print(sorted(catalog_codes - history_codes))

print("\nNo histórico mas não no catálogo:")
print(sorted(history_codes - catalog_codes))

No catálogo mas sem histórico:
[]

No histórico mas não no catálogo:
[]


In [32]:
materials_catalog

,codigo,descricao
0,M01,Britas
1,M02,Areias
2,M03,Inertes
3,M04,Ladrilhos de calcário e granito
4,M05,Cantarias de calcário e granito
5,M06,Ladr. e cant. de calcário e granito
6,M07,Telhas cerâmicas
7,M08,Tijolos cerâmicos
8,M09,Produtos Cerâmicos Vermelhos
9,M10,Azulejos e Mosaicos
